In [2]:
import pandas as pd

DATA_PATH = "../data/raw/DataCoSupplyChainDataset.csv"

df = pd.read_csv(DATA_PATH, encoding="latin1")

In [3]:
columns = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk",
    "Product Status",
    "shipping date (DateOrders)",
    "Order State"
]

df[columns].head().style.hide(axis="index")

Days for shipping (real),Days for shipment (scheduled),Delivery Status,Late_delivery_risk,Product Status,shipping date (DateOrders),Order State
3,4,Advance shipping,0,0,2/3/2018 22:56,Java Occidental
5,4,Late delivery,1,0,1/18/2018 12:27,Rajastán
4,4,Shipping on time,0,0,1/17/2018 12:06,Rajastán
3,4,Advance shipping,0,0,1/16/2018 11:45,Queensland
2,4,Advance shipping,0,0,1/15/2018 11:24,Queensland


In [4]:
timeline_cols = [
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Days for shipment (scheduled)",
    "Days for shipping (real)",
    "Delivery Status",
    "Late_delivery_risk",
]

display(
    df[timeline_cols]
    .head(10)
    .style.hide(axis="index")
)

order date (DateOrders),shipping date (DateOrders),Days for shipment (scheduled),Days for shipping (real),Delivery Status,Late_delivery_risk
1/31/2018 22:56,2/3/2018 22:56,4,3,Advance shipping,0
1/13/2018 12:27,1/18/2018 12:27,4,5,Late delivery,1
1/13/2018 12:06,1/17/2018 12:06,4,4,Shipping on time,0
1/13/2018 11:45,1/16/2018 11:45,4,3,Advance shipping,0
1/13/2018 11:24,1/15/2018 11:24,4,2,Advance shipping,0
1/13/2018 11:03,1/19/2018 11:03,4,6,Shipping canceled,0
1/13/2018 10:42,1/15/2018 10:42,1,2,Late delivery,1
1/13/2018 10:21,1/15/2018 10:21,1,2,Late delivery,1
1/13/2018 10:00,1/16/2018 10:00,2,3,Late delivery,1
1/13/2018 9:39,1/15/2018 9:39,1,2,Late delivery,1


In [5]:
df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"], errors="coerce")
df["shipping date (DateOrders)"] = pd.to_datetime(df["shipping date (DateOrders)"], errors="coerce")
df["order_to_shipping_days"] = (df["shipping date (DateOrders)"] - df["order date (DateOrders)"]).dt.total_seconds() / 86400

df[
    [
        "order date (DateOrders)",
        "shipping date (DateOrders)",
        "order_to_shipping_days",
        "Days for shipment (scheduled)",
        "Days for shipping (real)",
        "Late_delivery_risk",
    ]
].head(10).style.hide(axis="index")

order date (DateOrders),shipping date (DateOrders),order_to_shipping_days,Days for shipment (scheduled),Days for shipping (real),Late_delivery_risk
2018-01-31 22:56:00,2018-02-03 22:56:00,3.000000,4,3,0
2018-01-13 12:27:00,2018-01-18 12:27:00,5.000000,4,5,1
2018-01-13 12:06:00,2018-01-17 12:06:00,4.000000,4,4,0
2018-01-13 11:45:00,2018-01-16 11:45:00,3.000000,4,3,0
2018-01-13 11:24:00,2018-01-15 11:24:00,2.000000,4,2,0
2018-01-13 11:03:00,2018-01-19 11:03:00,6.000000,4,6,0
2018-01-13 10:42:00,2018-01-15 10:42:00,2.000000,1,2,1
2018-01-13 10:21:00,2018-01-15 10:21:00,2.000000,1,2,1
2018-01-13 10:00:00,2018-01-16 10:00:00,3.000000,2,3,1
2018-01-13 09:39:00,2018-01-15 09:39:00,2.000000,1,2,1


In [6]:
(
    df["order_to_shipping_days"] == df["Days for shipping (real)"]
).value_counts()

True     170782
False      9737
Name: count, dtype: int64

In [7]:
# آیا target دقیقاً از real > scheduled ساخته شده؟
df["calculated_late"] = (
    df["Days for shipping (real)"]
    > df["Days for shipment (scheduled)"]
).astype(int)

pd.crosstab(
    df["calculated_late"],
    df["Late_delivery_risk"],
    margins=True
)

Late_delivery_risk,0,1,All
calculated_late,,,
0,77119,0,77119
1,4423,98977,103400
All,81542,98977,180519


In [8]:
exceptions = df[
    (df["Days for shipping (real)"] > df["Days for shipment (scheduled)"])
    & (df["Late_delivery_risk"] == 0)
]

exceptions["Delivery Status"].value_counts()

Delivery Status
Shipping canceled    4423
Name: count, dtype: int64

In [10]:
calculated_target = (
    (df["Days for shipping (real)"] > df["Days for shipment (scheduled)"])
    & (df["Delivery Status"] != "Shipping canceled")
).astype(int)

(calculated_target == df["Late_delivery_risk"]).value_counts()


True    180519
Name: count, dtype: int64

In [11]:
mismatch = df[
    df["order_to_shipping_days"] != df["Days for shipping (real)"]
]

mismatch[
    [
        "order date (DateOrders)",
        "shipping date (DateOrders)",
        "order_to_shipping_days",
        "Days for shipping (real)",
        "Delivery Status",
        "Late_delivery_risk",
    ]
].head(20)

,order date (DateOrders),shipping date (DateOrders),order_to_shipping_days,Days for shipping (real),Delivery Status,Late_delivery_risk
19,2018-01-13 06:09:00,2018-01-13 18:09:00,0.5,0,Shipping on time,0
20,2018-01-13 05:48:00,2018-01-13 17:48:00,0.5,0,Shipping on time,0
38,2018-01-12 23:30:00,2018-01-13 11:30:00,0.5,1,Late delivery,1
39,2018-01-12 23:09:00,2018-01-13 11:09:00,0.5,1,Shipping canceled,0
40,2018-01-12 22:48:00,2018-01-13 10:48:00,0.5,1,Late delivery,1
41,2018-01-12 22:27:00,2018-01-13 10:27:00,0.5,1,Late delivery,1
387,2018-01-12 11:14:00,2018-01-12 23:14:00,0.5,0,Shipping on time,0
388,2018-01-12 10:53:00,2018-01-12 22:53:00,0.5,0,Shipping on time,0
389,2018-01-12 10:32:00,2018-01-12 22:32:00,0.5,0,Shipping on time,0
447,2018-01-11 14:13:00,2018-01-12 02:13:00,0.5,1,Late delivery,1


In [12]:
df["calendar_shipping_days"] = (
    df["shipping date (DateOrders)"].dt.normalize()
    - df["order date (DateOrders)"].dt.normalize()
).dt.days

(
    df["calendar_shipping_days"]
    == df["Days for shipping (real)"]
).value_counts()

True    180519
Name: count, dtype: int64

In [13]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

01. Type
02. Days for shipping (real)
03. Days for shipment (scheduled)
04. Benefit per order
05. Sales per customer
06. Delivery Status
07. Late_delivery_risk
08. Category Id
09. Category Name
10. Customer City
11. Customer Country
12. Customer Email
13. Customer Fname
14. Customer Id
15. Customer Lname
16. Customer Password
17. Customer Segment
18. Customer State
19. Customer Street
20. Customer Zipcode
21. Department Id
22. Department Name
23. Latitude
24. Longitude
25. Market
26. Order City
27. Order Country
28. Order Customer Id
29. order date (DateOrders)
30. Order Id
31. Order Item Cardprod Id
32. Order Item Discount
33. Order Item Discount Rate
34. Order Item Id
35. Order Item Product Price
36. Order Item Profit Ratio
37. Order Item Quantity
38. Sales
39. Order Item Total
40. Order Profit Per Order
41. Order Region
42. Order State
43. Order Status
44. Order Zipcode
45. Product Card Id
46. Product Category Id
47. Product Description
48. Product Image
49. Product Name
50. Product

In [14]:
df["Order Status"].value_counts(dropna=False)

Order Status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64

In [15]:
pd.crosstab(
    df["Order Status"],
    df["Late_delivery_risk"],
    normalize="index"
).round(3)

Late_delivery_risk,0,1
Order Status,,
CANCELED,1.000,0.000
CLOSED,0.434,0.566
COMPLETE,0.425,0.575
ON_HOLD,0.444,0.556
PAYMENT_REVIEW,0.428,0.572
PENDING,0.421,0.579
PENDING_PAYMENT,0.425,0.575
PROCESSING,0.429,0.571
SUSPECTED_FRAUD,1.000,0.000


In [16]:
desc = pd.read_csv(
    "../data/raw/DescriptionDataCoSupplyChain.csv",
    encoding="latin1",
)

fields_to_check = [
    "Benefit per order",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
]

desc[
    desc["FIELDS"].isin(fields_to_check)
].style.hide(axis="index")

FIELDS,DESCRIPTION
Benefit per order,: Earnings per order placed
Order Item Profit Ratio,: Order Item Profit Ratio
Order Profit Per Order,: Order Profit Per Order


In [17]:
financial_cols = [
    "Benefit per order",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Sales",
    "Order Item Total",
    "Order Item Product Price",
    "Order Item Quantity",
    "Order Item Discount",
]

df[financial_cols].head(10)

,Benefit per order,Order Item Profit Ratio,Order Profit Per Order,Sales,Order Item Total,Order Item Product Price,Order Item Quantity,Order Item Discount
0,91.250000,0.29,91.250000,327.75,314.640015,327.75,1,13.110000
1,-249.089996,-0.80,-249.089996,327.75,311.359985,327.75,1,16.389999
2,-247.779999,-0.80,-247.779999,327.75,309.720001,327.75,1,18.030001
3,22.860001,0.08,22.860001,327.75,304.809998,327.75,1,22.940001
4,134.210007,0.45,134.210007,327.75,298.250000,327.75,1,29.500000
5,18.580000,0.06,18.580000,327.75,294.980011,327.75,1,32.779999
6,95.180000,0.33,95.180000,327.75,288.420013,327.75,1,39.330002
7,68.430000,0.24,68.430000,327.75,285.140015,327.75,1,42.610001
8,133.720001,0.48,133.720001,327.75,278.589996,327.75,1,49.160000
9,132.149994,0.48,132.149994,327.75,275.309998,327.75,1,52.439999


In [18]:
df[financial_cols].corr(numeric_only=True).round(3)

,Benefit per order,Order Item Profit Ratio,Order Profit Per Order,Sales,Order Item Total,Order Item Product Price,Order Item Quantity,Order Item Discount
Benefit per order,1.000,0.824,1.000,0.132,0.133,0.103,0.016,0.065
Order Item Profit Ratio,0.824,1.000,0.824,-0.002,-0.001,-0.002,0.001,-0.003
Order Profit Per Order,1.000,0.824,1.000,0.132,0.133,0.103,0.016,0.065
Sales,0.132,-0.002,0.132,1.000,0.990,0.790,0.106,0.617
Order Item Total,0.133,-0.001,0.133,0.990,1.000,0.782,0.105,0.499
Order Item Product Price,0.103,-0.002,0.103,0.790,0.782,1.000,-0.476,0.488
Order Item Quantity,0.016,0.001,0.016,0.106,0.105,-0.476,1.000,0.065
Order Item Discount,0.065,-0.003,0.065,0.617,0.499,0.488,0.065,1.000


In [19]:
print(
    "Benefit == Order Profit:",
    (df["Benefit per order"] == df["Order Profit Per Order"]).value_counts()
)

print(
    "\nSales == Order Item Total:",
    (df["Sales"] == df["Order Item Total"]).value_counts()
)

print(
    "\nPossible profit formula check:"
)

calculated_profit_ratio = (
    df["Order Profit Per Order"] / df["Order Item Total"]
)

print(
    (calculated_profit_ratio.round(6)
     == df["Order Item Profit Ratio"].round(6))
    .value_counts()
)

Benefit == Order Profit: True    180519
Name: count, dtype: int64

Sales == Order Item Total: False    170491
True      10028
Name: count, dtype: int64

Possible profit formula check:
False    164528
True      15991
Name: count, dtype: int64


In [20]:
check = df[
    [
        "Sales",
        "Order Item Total",
        "Order Item Discount",
        "Order Item Discount Rate",
        "Order Item Product Price",
        "Order Item Quantity",
    ]
].copy()

check["sales_minus_total"] = (
    check["Sales"] - check["Order Item Total"]
)

check.head(10)

,Sales,Order Item Total,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Quantity,sales_minus_total
0,327.75,314.640015,13.110000,0.04,327.75,1,13.109985
1,327.75,311.359985,16.389999,0.05,327.75,1,16.390015
2,327.75,309.720001,18.030001,0.06,327.75,1,18.029999
3,327.75,304.809998,22.940001,0.07,327.75,1,22.940002
4,327.75,298.250000,29.500000,0.09,327.75,1,29.500000
5,327.75,294.980011,32.779999,0.10,327.75,1,32.769989
6,327.75,288.420013,39.330002,0.12,327.75,1,39.329987
7,327.75,285.140015,42.610001,0.13,327.75,1,42.609985
8,327.75,278.589996,49.160000,0.15,327.75,1,49.160004
9,327.75,275.309998,52.439999,0.16,327.75,1,52.440002


In [25]:
(
    ((df["Sales"] - df["Order Item Discount"]) - df["Order Item Total"]).abs()
    <= 0.011
).value_counts()

True    180519
Name: count, dtype: int64

In [26]:
(
    (
        (df["Order Item Product Price"] * df["Order Item Quantity"])
        - df["Sales"]
    )
    .abs()
    <= 0.011
).value_counts()

True    180519
Name: count, dtype: int64

In [27]:
df["Shipping Mode"].value_counts(dropna=False)

Shipping Mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64

In [28]:
pd.crosstab(
    df["Shipping Mode"],
    df["Late_delivery_risk"],
    normalize="index"
).round(3)

Late_delivery_risk,0,1
Shipping Mode,,
First Class,0.047,0.953
Same Day,0.543,0.457
Second Class,0.234,0.766
Standard Class,0.619,0.381


In [29]:
df["Days for shipment (scheduled)"].value_counts(dropna=False).sort_index()

Days for shipment (scheduled)
0      9737
1     27814
2     35216
4    107752
Name: count, dtype: int64

In [30]:
pd.crosstab(
    df["Days for shipment (scheduled)"],
    df["Late_delivery_risk"],
    normalize="index"
).round(3)

Late_delivery_risk,0,1
Days for shipment (scheduled),,
0,0.543,0.457
1,0.047,0.953
2,0.234,0.766
4,0.619,0.381


In [31]:
pd.crosstab(
    df["Shipping Mode"],
    df["Days for shipment (scheduled)"]
)

Days for shipment (scheduled),0,1,2,4
Shipping Mode,,,,
First Class,0,27814,0,0
Same Day,9737,0,0,0
Second Class,0,0,35216,0
Standard Class,0,0,0,107752


In [34]:
print(df["Market"].value_counts(dropna=False))

Market
LATAM           51594
Europe          50252
Pacific Asia    41260
USCA            25799
Africa          11614
Name: count, dtype: int64


In [35]:
print(
    df["Order Region"]
    .value_counts(dropna=False)
)

Order Region
Central America    28341
Western Europe     27109
South America      14935
Oceania            10148
Northern Europe     9792
Southeast Asia      9539
Southern Europe     9431
Caribbean           8318
West of USA         7993
South Asia          7731
Eastern Asia        7280
East of USA         6915
West Asia           6009
US Center           5887
South of  USA       4045
Eastern Europe      3920
West Africa         3696
North Africa        3232
East Africa         1852
Central Africa      1677
Southern Africa     1157
Canada               959
Central Asia         553
Name: count, dtype: int64


In [33]:
pd.crosstab(
    df["Market"],
    df["Order Region"]
)

Order Region,Canada,Caribbean,Central Africa,Central America,Central Asia,East Africa,East of USA,Eastern Asia,Eastern Europe,North Africa,...,South Asia,South of USA,Southeast Asia,Southern Africa,Southern Europe,US Center,West Africa,West Asia,West of USA,Western Europe
Market,,,,,,,,,,,,,,,,,,,,,
Africa,0,0,1677,0,0,1852,0,0,0,3232,...,0,0,0,1157,0,0,3696,0,0,0
Europe,0,0,0,0,0,0,0,0,3920,0,...,0,0,0,0,9431,0,0,0,0,27109
LATAM,0,8318,0,28341,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Pacific Asia,0,0,0,0,553,0,0,7280,0,0,...,7731,0,9539,0,0,0,0,6009,0,0
USCA,959,0,0,0,0,0,6915,0,0,0,...,0,4045,0,0,0,5887,0,0,7993,0


In [36]:
pd.crosstab(
    df["Market"],
    df["Late_delivery_risk"],
    normalize="index"
).round(3)

Late_delivery_risk,0,1
Market,,
Africa,0.454,0.546
Europe,0.448,0.552
LATAM,0.456,0.544
Pacific Asia,0.450,0.550
USCA,0.452,0.548


In [37]:
pd.crosstab(
    df["Order Region"],
    df["Late_delivery_risk"],
    normalize="index"
).round(3)

Late_delivery_risk,0,1
Order Region,,
Canada,0.512,0.488
Caribbean,0.469,0.531
Central Africa,0.420,0.580
Central America,0.452,0.548
Central Asia,0.447,0.553
East Africa,0.441,0.559
East of USA,0.443,0.557
Eastern Asia,0.457,0.543
Eastern Europe,0.443,0.557


In [39]:
print(df["Category Name"].value_counts(dropna=False))


Category Name
Cleats                  24551
Men's Footwear          22246
Women's Apparel         21035
Indoor/Outdoor Games    19298
Fishing                 17325
Water Sports            15540
Camping & Hiking        13729
Cardio Equipment        12487
Shop By Sport           10984
Electronics              3156
Accessories              1780
Golf Balls               1475
Girls' Apparel           1201
Golf Gloves              1070
Trade-In                  974
Video Games               838
Children's Clothing       652
Women's Clothing          650
Baseball & Softball       632
Hockey                    614
Cameras                   592
Toys                      529
Golf Shoes                524
Pet Supplies              492
Crafts                    484
Garden                    484
DVDs                      483
Computers                 442
Golf Apparel              441
Hunting & Shooting        440
Music                     434
Consumer Electronics      431
Boxing & MMA              

In [40]:
print(df["Category Id"].nunique())

51


In [41]:
category_late_rate = (
    df.groupby("Category Name")["Late_delivery_risk"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

category_late_rate.head(15)

,count,mean
Category Name,,
Golf Bags & Carts,61,0.688525
Lacrosse,343,0.600583
Pet Supplies,492,0.589431
Cameras,592,0.581081
Strength Training,111,0.576577
As Seen on TV!,68,0.573529
Music,434,0.571429
Accessories,1780,0.569663
Fitness Accessories,309,0.569579


In [42]:
category_late_rate.tail(15)

,count,mean
Category Name,,
Cardio Equipment,12487,0.544967
Men's Footwear,22246,0.544862
Soccer,138,0.543478
Men's Clothing,208,0.543269
Video Games,838,0.542959
DVDs,483,0.536232
Women's Golf Clubs,181,0.535912
Hockey,614,0.535831
Kids' Golf Clubs,384,0.533854


In [44]:
df["order_hour"] = df["order date (DateOrders)"].dt.hour
df["order_dayofweek"] = df["order date (DateOrders)"].dt.dayofweek
df["order_month"] = df["order date (DateOrders)"].dt.month

In [45]:
print(df["order_hour"].value_counts().sort_index())
print(df["order_dayofweek"].value_counts().sort_index())
print(df["order_month"].value_counts().sort_index())

order_hour
0     7590
1     7559
2     7574
3     7440
4     7719
5     7591
6     7270
7     7518
8     7435
9     7505
10    7697
11    7497
12    7486
13    7578
14    7556
15    7472
16    7501
17    7573
18    7534
19    7554
20    7481
21    7398
22    7418
23    7573
Name: count, dtype: int64
order_dayofweek
0    25786
1    25622
2    25587
3    25752
4    25925
5    25901
6    25946
Name: count, dtype: int64
order_month
1     17979
2     14529
3     15919
4     15435
5     15976
6     15139
7     15922
8     15912
9     15489
10    12955
11    12500
12    12764
Name: count, dtype: int64


In [46]:
df["order_year"] = df["order date (DateOrders)"].dt.year

pd.crosstab(
    df["order_year"],
    df["order_month"]
)

order_month,1,2,3,4,5,6,7,8,9,10,11,12
order_year,,,,,,,,,,,,
2015,5322,4729,5362,5126,5357,5134,5299,5273,5140,5302,5235,5371
2016,5317,4894,5210,5097,5302,5054,5305,5334,5160,5398,5210,5269
2017,5217,4906,5347,5212,5317,4951,5318,5305,5189,2255,2055,2124
2018,2123,0,0,0,0,0,0,0,0,0,0,0


In [47]:
df["order date (DateOrders)"].min(), df["order date (DateOrders)"].max()

(Timestamp('2015-01-01 00:00:00'), Timestamp('2018-01-31 23:38:00'))

In [48]:
monthly_late_by_year = (
    df.groupby(["order_year", "order_month"])["Late_delivery_risk"]
    .agg(["count", "mean"])
    .reset_index()
)

monthly_late_by_year

,order_year,order_month,count,mean
0,2015,1,5322,0.541150
1,2015,2,4729,0.548530
2,2015,3,5362,0.547557
3,2015,4,5126,0.538432
4,2015,5,5357,0.550868
5,2015,6,5134,0.541099
6,2015,7,5299,0.554633
7,2015,8,5273,0.556799
8,2015,9,5140,0.566926
9,2015,10,5302,0.548095


In [49]:
pd.crosstab(
    df["order_dayofweek"],
    df["Late_delivery_risk"],
    normalize="index"
).round(3)

Late_delivery_risk,0,1
order_dayofweek,,
0,0.449,0.551
1,0.457,0.543
2,0.452,0.548
3,0.450,0.550
4,0.448,0.552
5,0.456,0.544
6,0.449,0.551


In [50]:
pd.crosstab(
    df["order_hour"],
    df["Late_delivery_risk"],
    normalize="index"
).round(3)

Late_delivery_risk,0,1
order_hour,,
0,0.485,0.515
1,0.467,0.533
2,0.470,0.530
3,0.483,0.517
4,0.485,0.515
5,0.473,0.527
6,0.491,0.509
7,0.471,0.529
8,0.491,0.509


In [51]:
pd.crosstab(
    df["order_hour"],
    df["Shipping Mode"],
    normalize="index"
).round(3)

Shipping Mode,First Class,Same Day,Second Class,Standard Class
order_hour,,,,
0,0.140,0.060,0.201,0.600
1,0.155,0.059,0.191,0.595
2,0.147,0.046,0.213,0.593
3,0.138,0.067,0.188,0.607
4,0.151,0.061,0.198,0.591
5,0.158,0.045,0.190,0.606
6,0.149,0.061,0.181,0.609
7,0.161,0.055,0.201,0.582
8,0.143,0.054,0.187,0.616


In [52]:
hour_mode_late = (
    df.groupby(["order_hour", "Shipping Mode"])["Late_delivery_risk"]
    .mean()
    .unstack()
    .round(3)
)

hour_mode_late

Shipping Mode,First Class,Same Day,Second Class,Standard Class
order_hour,,,,
0,0.963,0.000,0.741,0.386
1,0.955,0.000,0.792,0.394
2,0.956,0.000,0.764,0.382
3,0.956,0.000,0.758,0.399
4,0.942,0.000,0.787,0.367
5,0.959,0.000,0.771,0.378
6,0.968,0.000,0.784,0.367
7,0.963,0.000,0.751,0.382
8,0.943,0.000,0.761,0.376


In [53]:
pd.crosstab(df["Shipping Mode"], df["Late_delivery_risk"])

Late_delivery_risk,0,1
Shipping Mode,,
First Class,1301,26513
Same Day,5283,4454
Second Class,8229,26987
Standard Class,66729,41023


In [54]:
shipping_summary = pd.crosstab(
    df["Shipping Mode"],
    df["Late_delivery_risk"]
)

shipping_summary["total"] = shipping_summary.sum(axis=1)

shipping_summary["late_rate"] = (
    shipping_summary[1] / shipping_summary["total"]
).round(3)

shipping_summary

Late_delivery_risk,0,1,total,late_rate
Shipping Mode,,,,
First Class,1301,26513,27814,0.953
Same Day,5283,4454,9737,0.457
Second Class,8229,26987,35216,0.766
Standard Class,66729,41023,107752,0.381


In [55]:
shipping_summary = pd.crosstab(
    df["Shipping Mode"],
    df["Late_delivery_risk"]
)

shipping_summary["total"] = shipping_summary.sum(axis=1)

shipping_summary["late_rate"] = (
    shipping_summary[1] / shipping_summary["total"]
).round(3)

shipping_summary

Late_delivery_risk,0,1,total,late_rate
Shipping Mode,,,,
First Class,1301,26513,27814,0.953
Same Day,5283,4454,9737,0.457
Second Class,8229,26987,35216,0.766
Standard Class,66729,41023,107752,0.381


In [56]:
same_day = df[df["Shipping Mode"] == "Same Day"]

pd.crosstab(
    same_day["order_hour"],
    same_day["Days for shipping (real)"]
)

Days for shipping (real),0,1
order_hour,,
0,452,0
1,446,0
2,350,0
3,500,0
4,468,0
5,343,0
6,443,0
7,416,0
8,403,0


In [57]:
same_day = df[df["Shipping Mode"] == "Same Day"].copy()

same_day["exact_shipping_hours"] = (
    same_day["shipping date (DateOrders)"]
    - same_day["order date (DateOrders)"]
).dt.total_seconds() / 3600

same_day["exact_shipping_hours"].value_counts().sort_index()

exact_shipping_hours
12.0    9737
Name: count, dtype: int64

In [58]:
ambiguous_cols = [
    "Benefit per order",
    "Sales per customer",
    "Latitude",
    "Longitude",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Order Status",
    "Product Description",
    "Product Status",
]

summary = pd.DataFrame({
    "dtype": df[ambiguous_cols].dtypes,
    "missing": df[ambiguous_cols].isna().sum(),
    "missing_pct": (df[ambiguous_cols].isna().mean() * 100).round(2),
    "unique_count": df[ambiguous_cols].nunique(dropna=False),
})

summary

,dtype,missing,missing_pct,unique_count
Benefit per order,float64,0,0.0,21998
Sales per customer,float64,0,0.0,2927
Latitude,float64,0,0.0,11250
Longitude,float64,0,0.0,4487
Order Item Profit Ratio,float64,0,0.0,162
Order Profit Per Order,float64,0,0.0,21998
Order Status,str,0,0.0,9
Product Description,float64,180519,100.0,1
Product Status,int64,0,0.0,1


In [62]:
df[df["Product Status"] == 1].value_counts()

Series([], Name: count, dtype: int64)

In [63]:
cols_to_check = [
    "Sales per customer",
    "Latitude",
    "Longitude",
    "Order Item Profit Ratio",
    "Benefit per order",
    "Order Status",
]

for col in cols_to_check:
    print(f"\n--- {col} ---")
    print(df[col].drop_duplicates().head(10).tolist())


--- Sales per customer ---
[314.6400146, 311.3599854, 309.7200012, 304.8099976, 298.25, 294.980011, 288.4200134, 285.1400146, 278.5899963, 275.3099976]

--- Latitude ---
[18.2514534, 18.27945137, 37.29223251, 34.12594605, 18.25376892, 43.01396942, 18.24253845, 25.92886925, 18.23322296, 37.77399063]

--- Longitude ---
[-66.03705597, -66.0370636, -121.881279, -118.2910156, -66.03704834, -78.87906647, -80.16287231, -121.966629, -73.58707428, -121.656517]

--- Order Item Profit Ratio ---
[0.289999992, -0.800000012, 0.079999998, 0.449999988, 0.059999999, 0.330000013, 0.239999995, 0.479999989, 0.170000002, 0.100000001]

--- Benefit per order ---
[91.25, -249.0899963, -247.7799988, 22.86000061, 134.2100067, 18.57999992, 95.18000031, 68.43000031, 133.7200012, 132.1499939]

--- Order Status ---
['COMPLETE', 'PENDING', 'CLOSED', 'PENDING_PAYMENT', 'CANCELED', 'PROCESSING', 'SUSPECTED_FRAUD', 'ON_HOLD', 'PAYMENT_REVIEW']


In [64]:
df[
    [
        "Latitude",
        "Longitude",
        "Customer City",
        "Customer State",
        "Customer Country",
        "Order City",
        "Order State",
        "Order Country",
    ]
].head(20)

,Latitude,Longitude,Customer City,Customer State,Customer Country,Order City,Order State,Order Country
0,18.251453,-66.037056,Caguas,PR,Puerto Rico,Bekasi,Java Occidental,Indonesia
1,18.279451,-66.037064,Caguas,PR,Puerto Rico,Bikaner,Rajastán,India
2,37.292233,-121.881279,San Jose,CA,EE. UU.,Bikaner,Rajastán,India
3,34.125946,-118.291016,Los Angeles,CA,EE. UU.,Townsville,Queensland,Australia
4,18.253769,-66.037048,Caguas,PR,Puerto Rico,Townsville,Queensland,Australia
5,43.013969,-78.879066,Tonawanda,NY,EE. UU.,Toowoomba,Queensland,Australia
6,18.242538,-66.037056,Caguas,PR,Puerto Rico,Guangzhou,Guangdong,China
7,25.928869,-80.162872,Miami,FL,EE. UU.,Guangzhou,Guangdong,China
8,18.233223,-66.037056,Caguas,PR,Puerto Rico,Guangzhou,Guangdong,China
9,37.773991,-121.966629,San Ramon,CA,EE. UU.,Guangzhou,Guangdong,China


In [65]:
checks = pd.DataFrame({
    "equals_sales": df["Sales per customer"].eq(df["Sales"]),
    "equals_order_item_total": df["Sales per customer"].eq(df["Order Item Total"]),
})

checks.sum()

equals_sales                10028
equals_order_item_total    180519
dtype: int64